![MOMENTUM · Hackatón de DRL · 01 Entornos](../img/banner-e01.jpg)

# **1. Introducción a los entornos de Gymnasium**

La librería [`gymnasium`](https://gymnasium.farama.org/index.html) proporciona una interfaz estándar para conectar agentes de **aprendizaje por refuerzo** (RL) con entornos de simulación. De esta forma, el agente puede observar el entorno, realizar acciones y recibir información sobre sus consecuencias siguiendo siempre las mismas reglas, independientemente de cómo estén implementados.

En este cuaderno, vamos a trabajar con [`highway-env`](https://highway-env.farama.org/), una colección de entornos de simulación sobre **conducción autónoma**. Nuestro agente deberá aprender a conducir en diferentes escenarios, percibiendo información de la calzada y del resto de vehículos. El objetivo no es todavía entrenar un agente sofisticado, sino entender qué es un entorno, cómo se inicializa, cómo se observa su estado y cómo se ejecutan acciones.

Comenzamos importando las librerías mencionadas, así como otras utilidades que emplearemos más adelante:

In [ ]:
import gymnasium as gym
import highway_env

import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

## **1.1. Definiendo el entorno**

A continuación, vamos a crear un entorno de simulación. Dicho entorno es el "mundo" en el cual nuestro agente se desenvuelve.

La librería `highway-env`, proporciona [múltiples entornos](https://highway-env.farama.org/environments/) de conducción autónoma con diferentes objetivos:

* **Highway:** conducir por una autopista evitando colisiones y priorizando circular por la derecha.
* **Merge:** facilitar la incorporación de otro vehículo a una autopista con tráfico.
* **Roundabout:** circular por una rotonda con tráfico.
* **Parking:** aparcar en una plaza de _parking_.
* **Intersection:** cruzar una intersección sin semáforos.
* **Racetrack:** recorrer un circuito manteniéndose en la carretera.
* **Lane Keeping:** mantener el vehículo dentro del carril.
* **Two Way:** adelantar con tráfico en sentido contrario.
* **Exit:** cambiar de carril para tomar una salida.
* **U-Turn:** adelantar mediante un giro en U.

En nuestro caso, utilizaremos el entorno [Highway](https://highway-env.farama.org/environments/highway/) (`highway-v0`). 

Con la función `make`, indicamos el entorno a crear y su configuración:

In [ ]:
env = gym.make("highway-v0", render_mode="rgb_array")

Una vez creado el entorno, podemos visualizar el **estado inicial** de la simulación:

In [ ]:
# Inicializar el entorno
env.reset()

# Mostrar estado inicial del entorno
show_state(env)

Observamos los diferentes carriles de la calzada y varios vehículos. El **vehículo verde** es el controlado por nuestro agente, mientras que los **vehículos azules** actúan de forma independiente.

Nuestro **objetivo** será evitar colisiones con el resto de vehículos, circulando a una velocidad elevada y manteniendo el vehículo por el carril de más a la derecha, siempre y cuando sea posible.

## **1.2. Estados y acciones**

Vamos a definir una serie de conceptos clave que nos ayudarán a entender la interacción entre agente y entorno:

- **Estado**: información que describe la situación actual del entorno y que el agente utiliza para decidir qué hacer.
- **Acción**: decisión o movimiento que el agente puede realizar en un determinado estado.

El conjunto de posibles estados de un entorno se denomina **espacio de estados**, mientras que el conjunto de posibles acciones se denomina **espacio de acciones**.

Observemos el espacio de acciones de nuestro entorno:

In [ ]:
env.action_space

Accediendo al atributo `action_space`, observamos que se trata de un **espacio de acciones discreto**, compuesto por **cinco posibles valores**. Es decir, en cada instante de tiempo, nuestro agente decidirá una de cinco posibles acciones.

Si consultamos la [documentación](https://highway-env.farama.org/actions/#discrete-actions), las acciones que nuestro agente puede tomar son las siguientes (cada una representada por un número):

- `0`: cambiar al carril izquierdo.
- `1`: mantener velocidad y carril actuales.
- `2`: cambiar al carril derecho.
- `3`: acelerar.
- `4`: reducir velocidad.

Aunque lo veremos en detalle más adelante, podemos "ejecutar" acciones utilizando la función `step`. Esta función recibe como parámetro la acción a realizar por el agente, y modifica el entorno en consecuencia.

Por ejemplo, si decidimos que nuestro agente cambie al carril izquierdo, utilizamos la acción `0` de la siguiente forma:

In [ ]:
# Aplicar accion
env.step(0)

# Mostrar imagen del entorno
show_state(env)

La imagen muestra el resultado de aplicar la acción sobre el estado actual del entorno. En el entorno por defecto, el agente nunca puede salir de la calzada.

Pasemos ahora a analizar en qué consiste el **espacio de estados** de este entorno:

* Todo estado en `highway-v0` se representa con una matriz $V \times F$ que contiene información de los $V$ vehículos más cercanos, mediante un conjunto de características de tamaño $F$. 

Por ejemplo:

<div style="text-align: left;">

| Vehículo   | Posición X | Posición Y | Velocidad X | Velocidad Y |
|------------|-----------:|-----------:|------------:|------------:|
| agente     | 5.0        | 4.0        | 15.0        | 0           |
| vehículo 1 | -10.0      | 4.0        | 12.0        | 0           |
| vehículo 2 | 13.0       | 8.0        | 13.5        | 0           |
| …          | …          | …          | …           | …           |
| vehículo V | 22.2       | 10.5       | 18.0        | 0.5         |

</div>

- **Vehículo**: indica el vehículo del que se recoge información. La información correspondiente al vehículo conducido por el agente siempre ocupa la primera fila. El resto de filas corresponden con el resto de vehículos no controlados.
- **Posición X**, **Posición Y**: coordenadas indicando la posición del vehículo.
- **Velocidad X**, **Velocidad Y**: velocidad en los ejes X e Y del vehículo.

Veamos en qué consiste el espacio de estados (`observation_space`):

In [ ]:
env.observation_space

Lo que nos indica la salida es que se trata de un espacio de estados de tipo [`Box`](https://gymnasium.farama.org/api/spaces/fundamental/#gymnasium.spaces.Box) (un tipo de espacio que representa valores numéricos continuos dentro de unos límites).

* Es decir, cada observación es una **matriz** de dimensiones $5 \times 5$ con valores en $(-\infty, \infty)$.

Anteriormente hemos visto cómo utilizar la función `step` para ejecutar acciones en el entorno. Pues bien, dicha función no solo ejecuta una acción, sino que devuelve como resultado información sobre el nuevo estado alcanzado (entre otras cosas que veremos más adelante).

Veamos el nuevo estado alcanzado tras ejecutar la acción `2` (cambiar al carril derecho):

In [ ]:
state, _, _, _, _ = env.step(2) # De toda la informacion devuelta por step, solamente nos interesa el estado

state

Podemos ver cómo `step` devuelve el estado resultante tras aplicar la acción, y cómo dicho estado tiene un formato similar al que hemos descrito anteriormente.

Dicho estado se corresponde con la siguiente imagen:

In [ ]:
# Mostrar imagen del entorno
show_state(env)

## **1.3. Recompensa, terminación y truncado**

Ya hemos visto cómo ejecutar acciones sobre el entorno empleando `step`, y cómo dichas acciones dan lugar a nuevos estados. Esta interacción agente-entorno se repite de forma repetida a lo largo del tiempo, dando lugar a un **episodio**.

- **Episodio**: secuencia de estados y acciones que comienza en un **estado inicial** y termina cuando se alcanza un **estado terminal** o se cumple una condición de finalización, como un límite de tiempo.

En nuestro caso, el entorno `highway-v0` cuenta con dos posibles condiciones de finalización:

- **Terminación** (`terminated = True`). Ocurre cuando el vehículo colisiona.
- **Truncado** (`truncated = True`). Se produce al alcanzar un límite de tiempo (por defecto, 40 segundos).

Por otro lado, para que un agente de RL pueda aprender una política de comportamiento óptima, es necesario definir una **función de recompensa**.

-  **Recompensa**: valor numérico que recibe el agente después de realizar una acción y que indica qué tan favorable ha sido esa acción para alcanzar el objetivo del entorno.

La **recompensa** es otro valor devuelto por la función `step`, junto al estado y las condiciones de terminación/truncado. 

Veamos un ejemplo aplicando la acción `1` (mantener velocidad y dirección) sobre el entorno:

In [ ]:
state, reward, terminated, truncated, _ = env.step(1)

# Mostrar imagen del entorno
show_state(env)

print("Recompensa obtenida: ", reward)
print("¿Terminar episodio? ", terminated)
print("¿Truncar episodio? ", truncated)

Al no alcanzarse aún el límite de tiempo, ni haber una colisión, `terminated` y `truncated` tienen valor `False`. 

Con respecto a la recompensa, en `highway-v0`, la [función de recompensa](https://highway-env.farama.org/rewards/#rewards) asigna un mayor valor a las acciones que:

1. Evitan colisiones.
2. Mantienen una velocidad elevada.
3. Priorizan la circulación por los carriles de la derecha (pequeña bonificación).

Esta recompensa se representa mediante la siguiente función:

$
R(s,a) = a\frac{v - v_{\min}}{v_{\max} - v_{\min}} + b\,l - c\,\text{collision}
$

donde:

* $v, v_\text{min}, v_\text{max}$ representan, respectivamente, la velocidad actual, mínima y máxima del vehículo del agente.
* $l$ representa la posición normalizada del carril, con valores entre 0 (carril más a la izquierda) y 1 (carril más a la derecha).
* $\text{collision}$ indica si se ha producido una colisión.
* $a$, $b$ y $c$ son coeficientes que determinan la importancia relativa de los términos de velocidad, carril y colisión.

En otras palabras, la recompensa aumenta con la velocidad y al circular por carriles situados más a la derecha, mientras que disminuye cuando se produce una colisión.

Por ejemplo, si a partir del estado anterior decidimos acelerar, la recompensa aumenta:

In [ ]:
_, reward, _, _, _ = env.step(3) # Acelerar

print("Recompensa obtenida: ", reward)

Finalmente, la función `step` devuelve un último valor: `info`. Se trata de un diccionario con información adicional, no percibida diretamente por el agente.

En `highway-v0`, dicho diccionario contiene la siguiente información:

* `speed`: velocidad actual del vehículo del agente (en $m/s$).
* `crashed`: indica si el vehículo ha sufrido una colisión.
* `action`: última acción realizada por el agente.
* `rewards`: contiene el desglose de los diferentes componentes de la recompensa.

Veamos un ejemplo:


In [ ]:
_, _, _, _, info = env.step(1)

info

## **1.4. Interacción agente-entorno**

Una vez definidos los componentes principales que definen la interacción agente-entorno, veamos el código necesario para simular un episodio.

A continuación, se muestra un ejemplo de agente que siempre mantiene la misma dirección y velocidad (`action = 1`). Puedes probar a modificar `action` y ver cómo se ve alterado el comportamiento del agente.

In [ ]:
state, info = env.reset() # La funcion reset devuelve el estado inicial y el diccionario info
terminated = truncated = False

# Bucle de interaccion agente-entorno (1 episodio)
while not (terminated or truncated):
    action = 1 # Mantener velocidad y direccion
    state, reward, terminated, truncated, info = env.step(action)
    
    # Mostrar imagen del entorno
    show_state(env)
    
env.close() # Es recomendable cerrar el entorno una vez terminada la ejecucion

Finalmente, si queremos ejecutar varios episodios, introducimos el código anterior dentro de un bucle `for`, tal que:

In [ ]:
N_EPISODES = 3 # Numero de episodios a ejecutar

for i in range(N_EPISODES):
    state, _ = env.reset()

    terminated = truncated = False

    while not (terminated or truncated):
        action = env.action_space.sample() # Devuelve una accion aleatoria del espacio de estados
        state, reward, terminated, truncated, _ = env.step(action)

        show_state(env)
    
env.close()

¡Ya contamos con un entorno listo para empezar a experimentar!

En este cuaderno, hemos aprendido cómo crear un entorno y los componentes básicos que definen un problema de RL. También hemos aprendido a definir un bucle de interacción agente-entorno y cómo ejecutar múltiples episodios de simulación.

En el próximo cuaderno, veremos cómo entrenar agentes inteligentes que aprendan a través de la interacción con el entorno y optimicen la función de recompensa.